In [ ]:
!pip install tensorflow

  Obtaining dependency information for tensorflow from https://files.pythonhosted.org/packages/6d/69/9999c2d9e8a3b08dfcfc7e9259a05fb1da5f700936091d2eb4a7985c2776/tensorflow-2.16.2-cp311-cp311-macosx_10_15_x86_64.whl.metadata
  Using cached tensorflow-2.16.2-cp311-cp311-macosx_10_15_x86_64.whl.metadata (4.1 kB)
  Obtaining dependency information for astunparse>=1.6.0 from https://files.pythonhosted.org/packages/2b/03/13dde6512ad7b4557eb792fbcf0c653af6076b81e5941d36ec61f7ce6028/astunparse-1.6.3-py2.py3-none-any.whl.metadata
  Using cached astunparse-1.6.3-py2.py3-none-any.whl.metadata (4.4 kB)
  Obtaining dependency information for flatbuffers>=23.5.26 from https://files.pythonhosted.org/packages/b8/25/155f9f080d5e4bc0082edfda032ea2bc2b8fab3f4d25d46c1e9dd22a1a89/flatbuffers-25.2.10-py2.py3-none-any.whl.metadata
  Using cached flatbuffers-25.2.10-py2.py3-none-any.whl.metadata (875 bytes)
  Obtaining dependency information for gast!=0.5.0,!=0.5.1,!=0.5.2,>=0.2.1 from https://files.pythonho

In [ ]:
# -*- coding: utf-8 -*-
"""
超軽量モデルの中間層特徴量視覚化デモ (MobileNetV3-Small)

TensorFlow HubからMobileNetV3-Smallをロードし、
指定した中間層の出力（特徴マップ）を視覚化します。
"""

import tensorflow as tf
import tensorflow_hub as hub
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import requests
from io import BytesIO

# --- パラメータ設定 ---
# 使用するモデル (TensorFlow HubのURL)
# MobileNetV3 Small (1.0 float, 224x224)
# 他のモデルも試せます: https://tfhub.dev/google/collections/imagenet/1
model_url = "https://tfhub.dev/google/imagenet/mobilenet_v3_small_100_224/feature_vector/5"
# モデルの入力画像サイズ
img_height, img_width = 224, 224

# デモ用画像のURL (好きな画像URLに変更可能)
# 例: 猫の画像
# image_url = "https://storage.googleapis.com/download.tensorflow.org/example_images/cat.jpg"
# 例: 犬の画像
image_url = "https://storage.googleapis.com/download.tensorflow.org/example_images/grace_hopper.jpg"


# --- 1. モデルのロード ---
print(f"Loading model from: {model_url}")
# feature_vectorではなく、classificationモデルをロードして中間層にアクセスしやすくする
# include_top=False とすることで、最終的な全結合層を除いたモデルを取得
# input_shape を指定する
base_model = tf.keras.applications.MobileNetV3Small(
    input_shape=(img_height, img_width, 3),
    include_top=False, # 分類層は不要
    weights='imagenet' # ImageNetで事前学習済みの重みを使用
)
base_model.trainable = False # 重みは固定する
print("Model loaded successfully.")

# モデルの構造を確認 (オプション)
# base_model.summary()

# --- 2. 画像の準備 ---
print(f"Loading image from: {image_url}")
try:
    response = requests.get(image_url)
    response.raise_for_status() # エラーチェック
    img = Image.open(BytesIO(response.content)).convert('RGB')
except requests.exceptions.RequestException as e:
    print(f"Error downloading image: {e}")
    # エラーが発生した場合、代替の画像や処理を行うことができます
    # ここでは簡単なダミー画像を作成します
    img = Image.new('RGB', (img_width, img_height), color = 'red')
    print("Using a dummy red image instead.")


# 画像のリサイズと前処理
img_resized = img.resize((img_width, img_height))
img_array = tf.keras.preprocessing.image.img_to_array(img_resized)
# MobileNetV3の入力は0から1の範囲に正規化する
img_array = img_array / 255.0
# バッチ次元を追加 (1, height, width, channels)
img_batch = np.expand_dims(img_array, axis=0)

print(f"Image preprocessed: shape={img_batch.shape}, dtype={img_batch.dtype}, range=[{img_batch.min()}, {img_batch.max()}]")

# 元画像を表示
plt.figure(figsize=(4, 4))
plt.imshow(img_resized)
plt.title("Input Image")
plt.axis("off")
plt.show()

# --- 3. 中間層の選択 ---
# base_model.summary() を実行して層の名前を確認し、視覚化したい層を選ぶ
# 例: いくつかの畳み込み層やボトルネック層を選択
# 層の名前はモデル構造によって異なります。summary()で確認してください。
# MobileNetV3Smallの例 (名前はTensorFlowのバージョン等で変わる可能性あり)
layer_names = [
    'Conv_1',              # 比較的浅い層
    'expanded_conv_1/squeeze_excite/Conv_1', # Squeeze-and-Exciteブロック内の層
    'expanded_conv_3/project', # 中間層のボトルネック射影
    'expanded_conv_8/project', # より深い層のボトルネック射影
    'Conv_2'               # 最終畳み込み層に近い層 (GlobalAveragePooling前)
]

# 存在するレイヤー名のみをフィルタリング
available_layer_names = [layer.name for layer in base_model.layers]
valid_layer_names = [name for name in layer_names if name in available_layer_names]

if not valid_layer_names:
    print("Warning: None of the specified layer names were found in the model.")
    print("Available layer names:", available_layer_names)
    # 代わりに利用可能な層からいくつか選ぶ (例)
    if len(available_layer_names) > 5:
         valid_layer_names = available_layer_names[1:len(available_layer_names):len(available_layer_names)//5][:5] # 適当に5つ選ぶ
    else:
         valid_layer_names = available_layer_names[1:] # 最初の層以外
    print(f"Using automatically selected layers instead: {valid_layer_names}")


# --- 4. 特徴量抽出モデルの作成 ---
layer_outputs = [base_model.get_layer(name).output for name in valid_layer_names]
feature_extractor_model = tf.keras.Model(inputs=base_model.input, outputs=layer_outputs)

# --- 5. 特徴量の抽出 ---
print("Extracting features...")
features = feature_extractor_model.predict(img_batch)
print("Features extracted.")

# --- 6. 特徴マップの視覚化 ---
print("Visualizing feature maps...")

for layer_name, feature_map_batch in zip(valid_layer_names, features):
    # バッチ次元を削除
    feature_map = feature_map_batch[0]
    n_features = feature_map.shape[-1] # チャネル数 (特徴マップの数)
    size = feature_map.shape[0]      # 特徴マップの高さ (幅も同じと仮定)

    print(f"\nLayer: {layer_name} (Shape: {feature_map.shape})")

    # 表示する特徴マップの数を制限 (多すぎると見づらいため)
    max_display_features = min(n_features, 64) # 最大64個まで表示
    display_grid = np.zeros((size, size * max_display_features))

    # 特徴マップをグリッドに配置
    for i in range(max_display_features):
        # i番目のチャネル (特徴マップ) を取得
        channel_image = feature_map[:, :, i]
        # 見やすいようにスケーリング (オプション)
        channel_image -= channel_image.mean()
        channel_image /= (channel_image.std() + 1e-5) # ゼロ除算を防ぐ
        channel_image *= 64
        channel_image += 128
        channel_image = np.clip(channel_image, 0, 255).astype('uint8')
        # グリッドに追加
        display_grid[:, i * size : (i + 1) * size] = channel_image

    # グリッドを表示
    scale = 20. / max_display_features
    plt.figure(figsize=(scale * max_display_features, scale))
    plt.title(f"Layer: {layer_name} ({n_features} features, showing {max_display_features})")
    plt.grid(False)
    plt.imshow(display_grid, aspect='auto', cmap='viridis') # viridisカラーマップを使用
    plt.axis('off')
    plt.show()

print("\nVisualization complete.")

ModuleNotFoundError: No module named 'tensorflow'